---
title: "Palmer Penguins Clustering"
subtitle: "Unsupervised Methods over Tiny Data using PySpark"
date: 03/12/2024
date-modified: today
date-format: long
engine: jupyter
toc: true
html:
    theme: [default, ../analytics_themes/web_analytics.scss]
categories: ["Spark", "PySpark", "SparkML"]
execute:
  eval: true
---

In [ ]:
import os
import sys
from pyspark.sql import SparkSession

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
spark = SparkSession.builder.config("spark.ui.enabled", "false") \
    .appName("palmerpenguins") \
    .config("spark.driver.host", "localhost") \
    .getOrCreate()

spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

# Load the CSV file into a Spark DataFrame
df = spark.read.option("header", "true").option("inferSchema", "true").option("multiLine","true").option("escape", "\"").option("nullValue", "NA").option("nanValue", "NaN").csv("./data/penguins.csv")

# 3. Register the DataFrame as a temporary SQL table
df.createOrReplaceTempView("palmerpenguins")

# size of the DataFrame

print(f"Size of the DataFrame: {df.count()} rows")

df.show(5)

Size of the DataFrame: 344 rows
+-------+---------+--------------+-------------+-----------------+-----------+------+----+
|species|   island|bill_length_mm|bill_depth_mm|flipper_length_mm|body_mass_g|   sex|year|
+-------+---------+--------------+-------------+-----------------+-----------+------+----+
| Adelie|Torgersen|          39.1|         18.7|              181|       3750|  male|2007|
| Adelie|Torgersen|          39.5|         17.4|              186|       3800|female|2007|
| Adelie|Torgersen|          40.3|         18.0|              195|       3250|female|2007|
| Adelie|Torgersen|          NULL|         NULL|             NULL|       NULL|  NULL|2007|
| Adelie|Torgersen|          36.7|         19.3|              193|       3450|female|2007|
+-------+---------+--------------+-------------+-----------------+-----------+------+----+
only showing top 5 rows


In [36]:
# df_clean = df.dropna()
df_clean = spark.sql("""
    SELECT species, bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g, sex
    from palmerpenguins
    """
)
df_clean.show(5)

+-------+--------------+-------------+-----------------+-----------+------+
|species|bill_length_mm|bill_depth_mm|flipper_length_mm|body_mass_g|   sex|
+-------+--------------+-------------+-----------------+-----------+------+
| Adelie|          39.1|         18.7|              181|       3750|  male|
| Adelie|          39.5|         17.4|              186|       3800|female|
| Adelie|          40.3|         18.0|              195|       3250|female|
| Adelie|          NULL|         NULL|             NULL|       NULL|  NULL|
| Adelie|          36.7|         19.3|              193|       3450|female|
+-------+--------------+-------------+-----------------+-----------+------+
only showing top 5 rows


In [37]:
# get the total data points per species
from narwhals import col
from pyspark.sql.functions import col

species_counts = df_clean.groupBy("species").count()
species_counts.show()

# get the total data points per species per sex
species_sex_counts = df_clean.groupBy("species", "sex").count()
# Show not null values only
# species_sex_counts = species_sex_counts.filter((col("species").isNotNull()) & (col("sex").isNotNull()))
# print(f"Size of the species/sex DataFrame: {species_sex_counts.count()} rows")
species_sex_counts.show()

+---------+-----+
|  species|count|
+---------+-----+
|   Gentoo|  124|
|   Adelie|  152|
|Chinstrap|   68|
+---------+-----+

+---------+------+-----+
|  species|   sex|count|
+---------+------+-----+
|   Gentoo|  NULL|    5|
|   Adelie|  NULL|    6|
|   Gentoo|female|   58|
|Chinstrap|  male|   34|
|Chinstrap|female|   34|
|   Gentoo|  male|   61|
|   Adelie|  male|   73|
|   Adelie|female|   73|
+---------+------+-----+



In [38]:
# Size of the cleaned DataFrame
print(f"Size of the cleaned DataFrame: {df_clean.count()} rows")

Size of the cleaned DataFrame: 344 rows


Looking at the proportion of the penguing we can see the data is heavily leaning towards Adelie. Also the male-female ratio seems to be around 1:1.

In [39]:
from pyspark.sql.functions import col, trim, lower

df_clustering = df.select(
    col('species'),
    col("bill_length_mm"),
    col("bill_depth_mm"),
    col("flipper_length_mm"),
    col("body_mass_g")
)
df_clustering = df_clustering.na.drop("any")
print(f"Size of the clustering DataFrame: {df_clustering.count()} rows")
df_clustering.show(5)


Size of the clustering DataFrame: 342 rows
+-------+--------------+-------------+-----------------+-----------+
|species|bill_length_mm|bill_depth_mm|flipper_length_mm|body_mass_g|
+-------+--------------+-------------+-----------------+-----------+
| Adelie|          39.1|         18.7|              181|       3750|
| Adelie|          39.5|         17.4|              186|       3800|
| Adelie|          40.3|         18.0|              195|       3250|
| Adelie|          36.7|         19.3|              193|       3450|
| Adelie|          39.3|         20.6|              190|       3650|
+-------+--------------+-------------+-----------------+-----------+
only showing top 5 rows


In [40]:
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType,DecimalType

df_clean.printSchema()

column_list = ["bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]

for column_name in column_list:
    df_clean = df_clean.withColumn(column_name, col(column_name).cast(DecimalType(8, 2))) # 123.12

df_clean.printSchema()

root
 |-- species: string (nullable = true)
 |-- bill_length_mm: double (nullable = true)
 |-- bill_depth_mm: double (nullable = true)
 |-- flipper_length_mm: integer (nullable = true)
 |-- body_mass_g: integer (nullable = true)
 |-- sex: string (nullable = true)

root
 |-- species: string (nullable = true)
 |-- bill_length_mm: decimal(8,2) (nullable = true)
 |-- bill_depth_mm: decimal(8,2) (nullable = true)
 |-- flipper_length_mm: decimal(8,2) (nullable = true)
 |-- body_mass_g: decimal(8,2) (nullable = true)
 |-- sex: string (nullable = true)



In [41]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

pdf = df_clean.toPandas()

for colname in ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]:
    fig = px.histogram(pdf, x=colname, color="species", barmode="overlay", marginal="box", nbins=60)
    fig.update_layout(title=f"{colname} Distribution by Species", template="simple_white")
    fig.show()

c:\Program Files\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [42]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

pdf = df.toPandas()

# Clean axis label titles
label_map = {
    "bill_length_mm": "Bill Length (mm)",
    "bill_depth_mm": "Bill Depth (mm)",
    "flipper_length_mm": "Flipper Length (mm)",
    "body_mass_g": "Body Mass (g)"
}

# Create 2x2 subplot grid
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"{label} by Species" for label in label_map.values()],
    horizontal_spacing=0.1,
    vertical_spacing=0.15
)
# Column pairs and subplot position
columns = list(label_map.keys())
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]


# Add histograms to each subplot
for i, (col, (row, col_num)) in enumerate(zip(columns, positions)):
    for species in pdf["species"].unique():
        subset = pdf[pdf["species"] == species]
        fig.add_trace(
            go.Histogram(
                x=subset[col],
                name=species,
                nbinsx=50,
                opacity=0.5,
                showlegend=True if i == 0 else False  # Only show legend once
            ),
            row=row, col=col_num
        )

    fig.update_xaxes(title_text=f"<b>{label_map[col]}</b>", row=row, col=col_num)
    fig.update_yaxes(title_text="<b>Count</b>", row=row, col=col_num)

fig.update_layout(
    title_text="Penguin Measurements Distribution by Species",
    height=650,
    template="simple_white",
    barmode='overlay'
)

fig.show()

c:\Program Files\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [43]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Clean axis label titles
label_map = {
    "bill_length_mm": "Bill Length (mm)",
    "bill_depth_mm": "Bill Depth (mm)",
    "flipper_length_mm": "Flipper Length (mm)",
    "body_mass_g": "Body Mass (g)"
}

# Create 2x2 subplot grid
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"{label} by Species" for label in label_map.values()],
    horizontal_spacing=0.1,
    vertical_spacing=0.15
)

columns = list(label_map.keys())
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

# Add histograms to each subplot
for col, (row, col_num) in zip(columns, positions):
    for species in pdf["species"].unique():
        subset = pdf[pdf["species"] == species]
        fig.add_trace(
            go.Histogram(
                x=subset[col],
                name=species,
                nbinsx=50,
                opacity=0.5,
                showlegend=True   # <-- legend ON for all traces
            ),
            row=row, col=col_num
        )

    fig.update_xaxes(title_text=f"<b>{label_map[col]}</b>", row=row, col=col_num)
    fig.update_yaxes(title_text="<b>Count</b>", row=row, col=col_num)

# Unified legend settings
fig.update_layout(
    title_text="Penguin Measurements Distribution by Species",
    height=650,
    template="simple_white",
    barmode='overlay',
    legend=dict(
        title="<b>Species</b>",
        tracegroupgap=0
    )
)

fig.show()


In [44]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

pdf = df.toPandas()
# Clean axis label titles
label_map = {
    "bill_length_mm": "Bill Length (mm)",
    "bill_depth_mm": "Bill Depth (mm)",
    "flipper_length_mm": "Flipper Length (mm)",
    "body_mass_g": "Body Mass (g)"
}

# Create 2x2 subplot grid
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"{label} by Species" for label in label_map.values()],
    horizontal_spacing=0.1,
    vertical_spacing=0.15
)
# Column pairs and subplot position
columns = list(label_map.keys())
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]


# Add histograms to each subplot
for i, (col, (row, col_num)) in enumerate(zip(columns, positions)):
    for species in pdf["species"].unique():
        subset = pdf[pdf["species"] == species]
        fig.add_trace(
            go.Histogram(
                x=subset[col],
                name=species,
                nbinsx=20,
                opacity=0.5,
                histnorm='probability density',
                showlegend=True if i == 0 else False  # Only show legend once
            ),
            row=row, col=col_num
        )

    fig.update_xaxes(title_text=f"<b>{label_map[col]}</b>", row=row, col=col_num)
    fig.update_yaxes(title_text="<b>Probability Density</b>", row=row, col=col_num)

fig.update_layout(
    title_text="Penguin Measurements Distribution by Species",
    height=800,
    template="simple_white",
    barmode='overlay'
)

fig.show()

c:\Program Files\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [45]:
pdf_renamed  = pdf.rename(columns=label_map)

# Feature pairs to plot in 2×2 grid (customized for nice visual contrast)
pairs = [
    ("bill_length_mm", "bill_depth_mm"),
    ("flipper_length_mm", "body_mass_g"),
    ("bill_length_mm", "flipper_length_mm"),
    ("bill_depth_mm", "body_mass_g"),
]

# Prepare the figure layout
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f"{label_map[x]} vs. {label_map[y]}" for x, y in pairs
    ],
    horizontal_spacing=0.08,
    vertical_spacing=0.12
)

# Plot each scatter
row_col = [(1, 1), (1, 2), (2, 1), (2, 2)]

species_list = pdf["species"].unique()
colors = px.colors.qualitative.Plotly

for idx, ((x, y), (r, c)) in enumerate(zip(pairs, row_col)):
    for species in species_list:
        species_data = pdf[pdf["species"] == species]
        fig.add_trace(
            go.Scatter(
                x=species_data[x],
                y=species_data[y],
                mode="markers",
                name=None,              # no legend entries
                marker=dict(color="black", size=6),
                showlegend=False        # suppress legend entirely
            ),
            row=r, col=c
        )

    fig.update_xaxes(title_text=f"<b>{label_map[x]}</b>", row=r, col=c)
    fig.update_yaxes(title_text=f"<b>{label_map[y]}</b>", row=r, col=c)

# Layout formatting
fig.update_layout(
    height=900,
    template="simple_white",
    title="<b>Pairwise Feature Scatterplots of Penguin Data</b>",
    title_x=0.5
)

fig.show()
fig.write_html("./output/pairwise_scatterplots.html")
fig.write_image("./output/pairwise_scatterplots.png", scale=2)
fig.write_image("./output/pairwise_scatterplots.svg")


In [46]:
pdf_renamed  = pdf.rename(columns=label_map)

# Feature pairs to plot in 2×2 grid (customized for nice visual contrast)
pairs = [
    ("bill_length_mm", "bill_depth_mm"),
    ("flipper_length_mm", "body_mass_g"),
    ("bill_length_mm", "flipper_length_mm"),
    ("bill_depth_mm", "body_mass_g"),
]

# Prepare the figure layout
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f"{label_map[x]} vs. {label_map[y]}" for x, y in pairs
    ],
    horizontal_spacing=0.08,
    vertical_spacing=0.12
)

# Plot each scatter
row_col = [(1, 1), (1, 2), (2, 1), (2, 2)]

species_list = pdf["species"].unique()
colors = px.colors.qualitative.Plotly

for idx, ((x, y), (r, c)) in enumerate(zip(pairs, row_col)):
    for i, species in enumerate(species_list):
        species_data = pdf[pdf["species"] == species]
        fig.add_trace(
            go.Scatter(
                x=species_data[x],
                y=species_data[y],
                mode="markers",
                name=species if (r, c) == (1, 1) else None,  # Avoid legend repetition
                marker=dict(color=colors[i], size=6),
                showlegend=(r, c) == (1, 1)
            ),
            row=r, col=c
        )
    # Update axis titles
    fig.update_xaxes(title_text=f"<b>{label_map[x]}</b>", row=r, col=c)
    fig.update_yaxes(title_text=f"<b>{label_map[y]}</b>", row=r, col=c)

# Layout formatting
fig.update_layout(
    height=900,
    width=1200,
    template="simple_white",
    title="<b>Pairwise Feature Scatterplots of Penguin Data</b>",
    title_x=0.5
)

fig.show()
fig.write_html("./output/pairwise_scatterplots_penguins.html")
fig.write_image("./output/pairwise_scatterplots_penguins.png", scale=2)
fig.write_image("./output/pairwise_scatterplots_penguins.svg")


In [47]:
from pyspark.sql.functions import col, trim, lower

df_clustering = df.select(
    col('species'),
    col("bill_length_mm"),
    col("bill_depth_mm"),
    col("flipper_length_mm"),
    col("body_mass_g")
)
df_clustering = df_clustering.na.drop("any")
df_clustering.show(5)


+-------+--------------+-------------+-----------------+-----------+
|species|bill_length_mm|bill_depth_mm|flipper_length_mm|body_mass_g|
+-------+--------------+-------------+-----------------+-----------+
| Adelie|          39.1|         18.7|              181|       3750|
| Adelie|          39.5|         17.4|              186|       3800|
| Adelie|          40.3|         18.0|              195|       3250|
| Adelie|          36.7|         19.3|              193|       3450|
| Adelie|          39.3|         20.6|              190|       3650|
+-------+--------------+-------------+-----------------+-----------+
only showing top 5 rows


In [48]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.types import DoubleType

print("Remaining rows:", df_clustering.count())
for col_name in df_clustering.columns:
    nulls = df_clustering.filter(col(col_name).isNull()).count()
    print(f"{col_name} nulls: {nulls}")


Remaining rows: 342
species nulls: 0
bill_length_mm nulls: 0
bill_depth_mm nulls: 0
flipper_length_mm nulls: 0
body_mass_g nulls: 0


$$
\begin{align}
\mathbb{P}(y|\mathbf{X}) = f(x_{1}, x_{2}, x_{3}, x_{4})\\
\mathbb{P}(\text{Penguin Species}|\text{bill length, bill depth, flipper length,	body mass,	sex}) = p\\
\mathbb{P}(\text{Penguin Species}=Gentoo|\text{bill length, bill depth, flipper length,	body mass,	sex}) = p_1\\
\mathbb{P}(\text{Penguin Species}=Chinstrap|\text{bill length, bill depth, flipper length,	body mass,	sex}) = p_2\\
\mathbb{P}(\text{Penguin Species}=Adelie|\text{bill length, bill depth, flipper length,	body mass,	sex}) = p_3\\
p_1+p_2+p_3 =100\\
\mathbb{P}(\text{Penguin Species}=Gentoo|\text{bill length} = 48.0) =53/124\\
\mathbb{P}(\text{Penguin Species}=Gentoo|\text{bill length} = 48.0, \text{bill depth}= ) =\frac{53}{124}\\
\end{align}
$$

In [49]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"],
    outputCol="features"
)
data = assembler.transform(df_clustering).select(["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "features", "species"])
data.show(5, truncate=False)

+--------------+-------------+-----------------+-----------+------------------------+-------+
|bill_length_mm|bill_depth_mm|flipper_length_mm|body_mass_g|features                |species|
+--------------+-------------+-----------------+-----------+------------------------+-------+
|39.1          |18.7         |181              |3750       |[39.1,18.7,181.0,3750.0]|Adelie |
|39.5          |17.4         |186              |3800       |[39.5,17.4,186.0,3800.0]|Adelie |
|40.3          |18.0         |195              |3250       |[40.3,18.0,195.0,3250.0]|Adelie |
|36.7          |19.3         |193              |3450       |[36.7,19.3,193.0,3450.0]|Adelie |
|39.3          |20.6         |190              |3650       |[39.3,20.6,190.0,3650.0]|Adelie |
+--------------+-------------+-----------------+-----------+------------------------+-------+
only showing top 5 rows


In [50]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
import plotly.graph_objects as go

scores = []
ks = list(range(2, 9))

for k in ks:
    kmeans = KMeans().setK(k).setSeed(42).setFeaturesCol("features")
    model = kmeans.fit(data)
    transformed = model.transform(data)

    evaluator = ClusteringEvaluator(
        featuresCol="features", predictionCol="prediction", metricName="silhouette")
    score = evaluator.evaluate(transformed)
    scores.append(score)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ks, y=scores, mode='lines+markers', name='Silhouette Score'))
fig.update_layout(title="Silhouette Score vs. k", 
                  xaxis_title="k",
                  yaxis_title="Score",
                  template="simple_white",
                  width=900, height=625)
# scale the axis from 0 to 0.80 with 0.10 increments
fig.update_yaxes(range=[0.60, 0.80], dtick=0.05)
fig.show()


In [51]:
#| echo: true
#| eval: true
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import SparkSession


# Create the KMeans model
kmeans = KMeans(k=3, seed=1, featuresCol="features", predictionCol="prediction")
model = kmeans.fit(data)

# Get predictions and show the results
predictions = model.transform(data)
predictions.show(5, truncate=False)


+--------------+-------------+-----------------+-----------+------------------------+-------+----------+
|bill_length_mm|bill_depth_mm|flipper_length_mm|body_mass_g|features                |species|prediction|
+--------------+-------------+-----------------+-----------+------------------------+-------+----------+
|39.1          |18.7         |181              |3750       |[39.1,18.7,181.0,3750.0]|Adelie |0         |
|39.5          |17.4         |186              |3800       |[39.5,17.4,186.0,3800.0]|Adelie |0         |
|40.3          |18.0         |195              |3250       |[40.3,18.0,195.0,3250.0]|Adelie |0         |
|36.7          |19.3         |193              |3450       |[36.7,19.3,193.0,3450.0]|Adelie |0         |
|39.3          |20.6         |190              |3650       |[39.3,20.6,190.0,3650.0]|Adelie |0         |
+--------------+-------------+-----------------+-----------+------------------------+-------+----------+
only showing top 5 rows


In [52]:
# Assign Species to a categorical variable for plotting purposes 0, 1, 2
predictions_df = predictions.toPandas()
pdf["species_code"] = pdf["species"].astype("category").cat.codes
predictions_df["species_code"] = predictions_df["species"].astype("category").cat.codes
# cross-tabulate species code and prediction
predictions_df.groupby(["species_code", "prediction"]).size().unstack()

c:\Program Files\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


prediction,0,1,2
species_code,,,
0,98.0,NaN,53.0
1,46.0,NaN,22.0
2,NaN,81.0,42.0


In [53]:
#| echo: false
#| eval: true
import plotly.express as px
import pandas as pd

# append columns bill_length_mm|bill_depth_mm|flipper_length_mm|body_mass_g from df_clustering to predictions
# predictions = predictions.join(df_clustering, on=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"], how="inner")

# Convert predictions to pandas for visualization
pandas_df = predictions.select("flipper_length_mm", "bill_depth_mm", "prediction", "species").toPandas()

# Plot using Plotly
fig = px.scatter(pandas_df, x="flipper_length_mm", y="bill_depth_mm", symbol="species", color="prediction",
                 title="K-Means Clustering on Penguins Dataset", 
                 labels={"flipper_length_mm": "Flipper Length", "bill_depth_mm": "Bill Depth"})

# update colors scale to be discrete and use Plotly qualitative colors
fig.update_traces(marker=dict(size=8, line=dict(width=1, color='DarkSlateGrey')), selector=dict(mode='markers'))
fig.update_layout(legend_title_text='Cluster', template="simple_white", width=900, height=600)

fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.3,  # Moves it below the x-axis
        xanchor="center",
        x=0.5,
    )
)

fig.show()


c:\Program Files\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [54]:
# ge the cross table of predictions and species
cross_tab = predictions.groupBy("prediction", "species").count().orderBy("prediction", "species")
cross_tab.show()

# Create a pandas datagframe and display confusion matrix
confusion_matrix = cross_tab.toPandas().pivot(index='prediction', columns='species', values='count').fillna(0)
print("Confusion Matrix:") 
print(confusion_matrix.head().to_string(index=False))

# Plot the confusion matrix as a heatmap using Plotly
import plotly.express as px
fig = px.imshow(confusion_matrix, text_auto=True, color_continuous_scale='Blues', labels=dict(x="Species", y="Predicted Cluster", color="Count"), title="Confusion Matrix of K-Means Clustering")
fig.update_layout(template="simple_white", width=600, height=500)



+----------+---------+-----+
|prediction|  species|count|
+----------+---------+-----+
|         0|   Adelie|   98|
|         0|Chinstrap|   46|
|         1|   Gentoo|   81|
|         2|   Adelie|   53|
|         2|Chinstrap|   22|
|         2|   Gentoo|   42|
+----------+---------+-----+



c:\Program Files\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Confusion Matrix:
 Adelie  Chinstrap  Gentoo
   98.0       46.0     0.0
    0.0        0.0    81.0
   53.0       22.0    42.0


In [55]:
from pyspark.ml.feature import PCA

pca = PCA(k=2, inputCol="features", outputCol="pcaFeatures")
pca_model = pca.fit(data)
projected = pca_model.transform(data)
pdf_proj = projected.toPandas()

pdf_proj["x"] = pdf_proj["pcaFeatures"].apply(lambda v: float(v[0]))
pdf_proj["y"] = pdf_proj["pcaFeatures"].apply(lambda v: float(v[1]))

fig = px.scatter(pdf_proj, x="x", y="y", color="species", title="PCA Projection Colored by Species")
fig.update_layout(template="simple_white",
                  width=900, 
                  height=700,
                  )
fig.update_xaxes(title_text="<b>Principal Component 1</b>")
fig.update_yaxes(title_text="<b>Principal Component 2</b>")
fig.show()


c:\Program Files\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [58]:
from pyspark.ml.clustering import GaussianMixture
from pyspark.sql import functions as F
from pyspark.ml.linalg import Vectors
from pyspark.ml.functions import vector_to_array
from pyspark.mllib.linalg import Vectors as OldVectors


gmm = GaussianMixture(k=3, seed=42, featuresCol="features")
gmm_model = gmm.fit(data)
gmm_pred = gmm_model.transform(data)
gmm_pred.show(5, truncate=False)

# Prob of data point 1 belonging to clusters is [0.99, 0.005, 0.005]
# change the probabilities column to a numeric 0, 1, 2 array of the max probability cluster

gmm_pred = gmm_pred.withColumn(
    "probability_array",
    vector_to_array("probability")
).withColumn(
    "cluster_argmax",
    F.expr("array_position(probability_array, array_max(probability_array)) - 1")
)

gmm_pred.select("features", "species", "probability", "prediction").show(5, truncate=False)


+--------------+-------------+-----------------+-----------+------------------------+-------+---------------------------------------------------------------+----------+
|bill_length_mm|bill_depth_mm|flipper_length_mm|body_mass_g|features                |species|probability                                                    |prediction|
+--------------+-------------+-----------------+-----------+------------------------+-------+---------------------------------------------------------------+----------+
|39.1          |18.7         |181              |3750       |[39.1,18.7,181.0,3750.0]|Adelie |[0.999999998168968,9.155160002553698E-10,9.155160002553698E-10]|0         |
|39.5          |17.4         |186              |3800       |[39.5,17.4,186.0,3800.0]|Adelie |[0.9999999990022692,4.98865369612063E-10,4.988653697916328E-10]|0         |
|40.3          |18.0         |195              |3250       |[40.3,18.0,195.0,3250.0]|Adelie |[0.999999998726516,6.367419675192764E-10,6.367419675228441E-10

In [ ]:
# Project GMM predictions
projected_gmm = pca_model.transform(gmm_pred)
pdf_gmm = projected_gmm.select("pcaFeatures", "prediction").toPandas()
pdf_gmm["x"] = pdf_gmm["pcaFeatures"].apply(lambda v: float(v[0]))
pdf_gmm["y"] = pdf_gmm["pcaFeatures"].apply(lambda v: float(v[1]))


c:\Program Files\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [ ]:

fig = px.scatter(pdf_gmm, x="x", y="y", color=pdf_gmm["prediction"].astype(str),
                 title="GMM Clustering on PCA-Projected Features")
fig.update_layout(template="simple_white",
                  width=900, 
                  height=700,)
fig.show()


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture

# Step 1: Fit GMM on 2D PCA-projected data
X_proj = pdf_gmm[["x", "y"]].values
gmm_model = GaussianMixture(n_components=len(pdf_gmm["prediction"].unique()), covariance_type='full', random_state=0)
gmm_model.fit(X_proj)

# Step 2: Create a mesh grid over x and y
x = np.linspace(X_proj[:, 0].min() - 1, X_proj[:, 0].max() + 1, 300)
y = np.linspace(X_proj[:, 1].min() - 1, X_proj[:, 1].max() + 1, 300)
xx, yy = np.meshgrid(x, y)
grid = np.c_[xx.ravel(), yy.ravel()]

# Step 3: Predict soft probabilities (posterior) over the mesh
probs = gmm_model.predict_proba(grid)
cluster_map = np.argmax(probs, axis=1).reshape(xx.shape)

# Step 4: Plot prediction area + data points
fig = go.Figure()

# Add contour for soft cluster regions
fig.add_trace(go.Contour(
    x=x, y=y, z=cluster_map,
    showscale=False,
    colorscale="Viridis",
    opacity=0.3,
    contours=dict(
        coloring="heatmap",
        showlines=False
    )
))

# Add data points
for label in sorted(pdf_gmm["prediction"].unique()):
    cluster_data = pdf_gmm[pdf_gmm["prediction"] == label]
    fig.add_trace(go.Scatter(
        x=cluster_data["x"], y=cluster_data["y"],
        mode="markers",
        name=f"Cluster {label}",
        marker=dict(size=5),
        hoverinfo="text",
        text=[f"Cluster: {label}" for _ in range(len(cluster_data))]
    ))

# Update layout
fig.update_layout(
    title="GMM Predicted Clusters with Region Overlay (PCA Projection)",
    xaxis_title="PCA 1",
    yaxis_title="PCA 2",
    template="simple_white",
    height=600,
    width=800
)

fig.show()


In [ ]:
import pandas as pd

# Palmer Penguins: Adelie = 0, Chinstrap = 1
penguins = pd.read_csv("penguins.csv") 
df = penguins[penguins['species'].isin(['Adelie', 'Chinstrap'])].copy()
df['label'] = (df['species'] == 'Chinstrap').astype(int)
df.dropna(subset=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'], inplace=True)
X = df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']].values
y = df['label'].values

df.head().style.hide(axis="index")

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year,label
Adelie,Torgersen,39.100000,18.700000,181.000000,3750.000000,male,2007,0
Adelie,Torgersen,39.500000,17.400000,186.000000,3800.000000,female,2007,0
Adelie,Torgersen,40.300000,18.000000,195.000000,3250.000000,female,2007,0
Adelie,Torgersen,36.700000,19.300000,193.000000,3450.000000,female,2007,0
Adelie,Torgersen,39.300000,20.600000,190.000000,3650.000000,male,2007,0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import numpy as np


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)
model = LogisticRegression()
model.fit(X_train, y_train)

ll = -log_loss(y_train, model.predict_proba(X_train), normalize=False)
n = X_train.shape[0]
k = X_train.shape[1] + 1
aic = -2 * ll + 2 * k
bic = -2 * ll + k * np.log(n)

print("Log-Likelihood:", ll)
print("AIC:", aic)
print("BIC:", bic)

Log-Likelihood: -4.380217324379187
AIC: 18.760434648758373
BIC: 34.58436451837595


In [ ]:
# # Regression curve
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib import cm
# from mpl_toolkits.mplot3d import Axes3D
# import celluloid
# from celluloid import Camera
# from scipy.special import expit
# from matplotlib import gridspec

# def pred_curve(xs,w0,b0):  # predict y-values for regression curve
#     z=xs.T@w0.T  + b0   
#     p=expit(z)  
#     return p

# # Create data for surface plot: 
# def CrossEntropy_cost(x,y,w0,b0):  # returns costs for every pair of w and b. 
#         p=expit(x @ w0.T + b0)
#         return - np.mean(y*np.log(p) + (1-y)*np.log(1-p))

# ws = np.linspace(-0.005, 0.075, 40) # set range of values for w ... 
# bs = np.linspace(-1.75, 0.75, 40)  # ... and b for surface plot
# M, B = np.meshgrid(ws, bs) # create meshgrid for surface plot
# zs = np.array([CrossEntropy_cost(X_train,y_train,       # determine costs for each pair of w and b 
#                        np.array([[wp]]), np.array([[bp]]))  # CrossEntropy_cost() only accepts wp and bp as matrices. 
#                for wp, bp in zip(np.ravel(M), np.ravel(B))])
# Z = zs.reshape(M.shape) # get z-values for surface plot in shape of M.

# # Define which epochs to plot: 
# a1=np.arange(0,20,1).tolist() 
# a2=np.arange(20,100,20).tolist()
# a3=np.arange(100,500,200).tolist()
# a4=np.arange(500,40000,4000).tolist()
# a5=np.arange(40000,600000,20000).tolist()
# points_ = a1+a2+a3+a4+a5

# # Create first animation (regression curve & surface plot):
# fig = plt.figure(figsize=(10,10)) # create figure
# gs = gridspec.GridSpec(1, 2, height_ratios=[1.7, 1]) # set height ratios of subplots

# label_font_size = 28 # size of label fonts
# tick_label_size= 18 # size of tick labels

# # Customize subplots: 
# ax0=fig.add_subplot(gs[0]) # add first subplot to figure
# ax0.set_title("Logistic regression curve", fontsize=22) #set title
# ax0.tick_params(axis='both', which='major', labelsize=tick_label_size) # set tick label size
# ax0.set_xlabel("x", fontsize=label_font_size, labelpad=5) # set axis labels
# ax0.set_ylabel("y", fontsize=label_font_size, labelpad=5)
# ax0.tick_params(axis='both', which='major',
#                 labelsize=tick_label_size) 
# ax0.set_ylim([-0.05, 1.2]) # set range for y-values in the plot
# ax0.set_yticks([0,0.2,0.4,0.6,0.8, 1]) # set y-axis ticks
# ax0.set_yticklabels(["0","0.2","0.4","0.6","0.8", "1.0"], # label y-ticks
#                     fontsize=tick_label_size)
# ax1=fig.add_subplot(gs[1], projection="3d") # add second subplot 
# ax1.view_init(elev=25, azim=-70) # set view
# ax1.set_xlabel("w", fontsize=label_font_size, labelpad=16) 
# ax1.set_ylabel("b", fontsize=label_font_size, labelpad=18)
# ax1.set_zlabel("costs", fontsize=label_font_size, labelpad=13)  
# ax1.tick_params(axis='both', which='major', labelsize=tick_label_size) 
# ax1.set_xticks([0, 0.02,0.04,0.06]) 
# ax1.set_xticklabels(["0", "0.02","0.04","0.06"], fontsize=tick_label_size)
# ax1.set_yticks([-1.5,-1, -0.5,0,0.5]) 
# ax1.set_yticklabels(["-1.5","-1", "-0.5","0","0.5"], fontsize=tick_label_size)
# ax1.set_zticks([0.4,0.6, 0.8,1.0]) 
# ax1.set_zticklabels(["0.4","0.6", "0.8","1.0"], fontsize=tick_label_size)
# ax1.tick_params(axis='z', pad=7, which='major', labelsize=tick_label_size) 
# plt.tight_layout() # adjusts subplots to figure area

# camera = Camera(fig)  # create camera 
# for i in points_:
    
#     # Plot logistic regression curve: 
#     ys = pred_curve(xs,np.array([w.flatten()[i]]),b.flatten()[i]) #calculate y-values for reg. curve
#     leg=ax0.plot(xs.flatten(),ys.flatten(),
#                  color='r', label=str(i))
    
#     # Scatter plot of training data:
#     ax0.scatter(X_train, y_train, color='b',marker='x', s=92) # plot training data points
#     ax0.legend(leg,[f'costs: {np.round(c.flatten()[i],3)}'], # customize legend
#                loc='upper left', fontsize=18)
    
#     # Plot connecting lines (between predicted y-values and training data points):
#     cl_=cl[i]
#     for j in range(len(y_train)):    
#         x,y = [X_train[j],X_train[j]],[float(y_train[j]),cl_[j]]
#         ax0.scatter(x,y, color='black',s=0.5)
#         ax0.plot(x,y, color='black', linewidth=1.5, 
#                  linestyle='dashed', alpha=0.4)
        
#     # Surface plot of costs:
#     ax1.plot_surface(M, B, Z, rstride=1, cstride=1, 
#                      alpha=0.80,cmap=cm.coolwarm) 
    
#     # Plot trajectory of gradient descent:
#     ax1.plot(w.flatten()[0:i],b.flatten()[0:i], 
#              c.flatten()[0:i], linestyle="dashed",
#              linewidth=2.5, color="black")
#     ax1.scatter(w.flatten()[i],b.flatten()[i],   
#                 c.flatten()[i], marker="o",s=70*2, color="black")
    
#     # Customize legend: 
#     ax1.legend(leg,[f'epochs: {i}'], 
#                loc='upper left', fontsize=18)
    
#     camera.snap() # take snapshot after each frame/iteration
    
# # Create animation: 
# animation = camera.animate(interval = 100, # set delay between frames in milliseconds.
#                           repeat = False, repeat_delay = 0) 
# # Save animation: 
# animation.save('LogReg_1.gif', writer = 'imagemagick') 

# #print("Final weights: "+ str(w.flatten()[i]))    # 0.03579113198262307
# #print("Final bias: "+ str(b[i]))                 # [-1.11258967]
# #print("Final costs: " + str(c[i]))               # [0.40058369]


In [ ]:
#| eval: false
# import seaborn as sns
# from sklearn.tree import DecisionTreeClassifier
# from sklearn import preprocessing
# import dtreeviz

# # Load and clean dataset
# df = sns.load_dataset("penguins").dropna()

# # Encode target
# label_encoder = preprocessing.LabelEncoder()
# df['species_encoded'] = label_encoder.fit_transform(df['species'])

# # Features and target
# X = df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]
# y = df['species_encoded']

# # Train model
# clf = DecisionTreeClassifier(max_depth=3)
# clf.fit(X, y)

# # dtreeviz 2.x+ API
# viz_model = dtreeviz.model(clf,
#                            X_train=X.values,  # Ensure these are numpy arrays
#                            y_train=y.values,
#                            feature_names=X.columns.tolist(),
#                            target_name='species',
#                            class_names=label_encoder.classes_.tolist()  
#                            )

# # Render visualization
# viz = viz_model.view()
# viz             # Interactive viewer
# viz.save("penguins_tree.svg")  # Save as SVG


In [ ]:
# get the prior probabilities for penguins
prior_probs = pdf['species'].value_counts(normalize=False)
prior_probs

species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64

In [ ]:
# prior prob for bill length given species
pdf.groupby('species')['bill_length_mm'].value_counts(normalize=False)
# table for conditional probabilities
cond_probs = pdf.groupby('species')['bill_length_mm'].value_counts(normalize=True)
cond_probs

species  bill_length_mm
Adelie   41.1              0.046358
         37.8              0.033113
         39.6              0.033113
         40.6              0.026490
         36.0              0.026490
                             ...   
Gentoo   48.1              0.008130
         51.5              0.008130
         55.1              0.008130
         48.8              0.008130
         49.9              0.008130
Name: proportion, Length: 208, dtype: float64